# Module 06 — Notebook 3 Solutions: Eval Script Mini-Project

In [ ]:
import sys
sys.path.insert(0, "../../../")
from src.checks import check_equal, check_type, check_approx, check_contains
import json
import subprocess
from pathlib import Path

SCRIPTS_DIR = Path("scripts")
SCRIPTS_DIR.mkdir(exist_ok=True)
DATA_JSON  = Path("../../../data/synthetic/model_outputs.json")
OUTPUT_DIR = Path("../../../output")
OUTPUT_DIR.mkdir(exist_ok=True)

if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))

## Exercise 1 Solution

In [ ]:
import importlib
import run_evaluation
importlib.reload(run_evaluation)

outputs   = run_evaluation.load_outputs(DATA_JSON)
summary   = run_evaluation.compute_summary(outputs, threshold=0.5)
n_flagged = summary["total_flagged"]
b1_rate   = round(float(summary["models"]["model-b-v1"]["flag_rate"]), 4)

In [ ]:
check_type(outputs, list, "outputs is a list")
check_equal(len(outputs), 20, "20 outputs loaded")
check_type(summary, dict, "summary is a dict")
check_equal(int(n_flagged), 7, "7 outputs flagged")
check_approx(b1_rate, 0.7778, 1e-3, "model-b-v1 flag rate")

## Exercise 2 Solution

In [ ]:
# subprocess.run is the programmatic equivalent of !python ... in notebooks
result = subprocess.run(
    ["python", "scripts/run_evaluation.py",
     "--input",     str(DATA_JSON),
     "--output",    str(OUTPUT_DIR / "ex2_results.json"),
     "--threshold", "0.3"],
    capture_output=True, text=True
)

exit_code = result.returncode
print(result.stdout)
print(result.stderr)

with open(OUTPUT_DIR / "ex2_results.json") as f:
    ex2_summary = json.load(f)
alerts = ex2_summary["alerts"]

In [ ]:
check_equal(int(exit_code), 0, "script exited with code 0")
check_type(ex2_summary, dict, "ex2_summary is a dict")
check_type(alerts, list, "alerts is a list")
check_equal(len(alerts) >= 1, True, "at least 1 alert at threshold 0.3")

## Exercise 3 Solution

In [ ]:
%%writefile scripts/run_evaluation_v2.py
"""run_evaluation_v2.py — adds --model filtering to run_evaluation.py"""
import argparse
import json
import logging
from collections import defaultdict
from pathlib import Path


def build_parser():
    parser = argparse.ArgumentParser(description="Evaluate model output flag rates.")
    parser.add_argument("--input",     type=Path, required=True)
    parser.add_argument("--output",    type=Path, default=Path("output/evaluation_results.json"))
    parser.add_argument("--threshold", type=float, default=0.5)
    parser.add_argument("--model",     type=str,  default=None, help="Filter to one model")
    parser.add_argument("--verbose",   action="store_true")
    return parser


def load_outputs(path):
    with open(path, encoding="utf-8") as f:
        return json.load(f)


def compute_summary(outputs, threshold):
    total   = len(outputs)
    flagged = sum(1 for o in outputs if o["flagged"])
    overall = flagged / total if total else 0.0
    counts  = defaultdict(lambda: {"total": 0, "flagged": 0})
    for o in outputs:
        counts[o["model"]]["total"]   += 1
        counts[o["model"]]["flagged"] += int(o["flagged"])
    logger  = logging.getLogger(__name__)
    models  = {}
    alerts  = []
    for model, c in sorted(counts.items()):
        rate = c["flagged"] / c["total"]
        models[model] = {"total": c["total"], "flagged": c["flagged"], "flag_rate": round(rate, 4)}
        if rate > threshold:
            msg = f"{model}: {rate:.1%} exceeds threshold {threshold:.0%}"
            logger.warning(msg)
            alerts.append(msg)
    return {"total_outputs": total, "total_flagged": flagged,
            "overall_flag_rate": round(overall, 4), "models": models, "alerts": alerts}


def save_results(summary, output_path):
    output_path.parent.mkdir(parents=True, exist_ok=True)
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(summary, f, indent=2)


def main(args):
    level = logging.DEBUG if args.verbose else logging.INFO
    logging.basicConfig(level=level, format="%(levelname)-8s %(message)s")
    logger = logging.getLogger(__name__)

    outputs = load_outputs(args.input)
    logger.info(f"Loaded {len(outputs)} outputs")

    # NEW: filter to a specific model if requested
    if args.model:
        outputs = [o for o in outputs if o["model"] == args.model]
        logger.info(f"Filtered to {len(outputs)} outputs for model '{args.model}'")

    summary = compute_summary(outputs, threshold=args.threshold)
    logger.info(f"Overall flag rate: {summary['overall_flag_rate']:.1%}")
    save_results(summary, args.output)
    logger.info("Done")
    return summary


if __name__ == "__main__":
    main(build_parser().parse_args())

In [ ]:
ex3_result = subprocess.run(
    ["python", "scripts/run_evaluation_v2.py",
     "--input",  str(DATA_JSON),
     "--output", str(OUTPUT_DIR / "ex3_results.json"),
     "--model",  "model-b-v1"],
    capture_output=True, text=True
)
print(ex3_result.stdout)
print(ex3_result.stderr)

with open(OUTPUT_DIR / "ex3_results.json") as f:
    ex3_data = json.load(f)
b1_only_rate = float(ex3_data["overall_flag_rate"])

In [ ]:
check_equal(ex3_result.returncode, 0, "v2 script exited cleanly")
check_approx(b1_only_rate, 0.7778, 1e-3, "model-b-v1 only flag rate is 0.7778")